# 110 — Anatomía: instrucciones, herramientas, estado y salida

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Anatomía de todo agente LLM — cuatro piezas:**

- **Instrucciones:** la política versionable — objetivo verificable, restricciones,
  procedimiento, formato de salida. Lo no escrito queda a criterio del modelo.
- **Herramientas:** contratos (nombre, descripción, JSON Schema). El modelo **emite
  intenciones** de llamada; el runtime valida y ejecuta — esa separación permite
  interponer permisos y aprobaciones.
- **Estado:** tres capas con vidas distintas — contexto (ventana, volátil y cara),
  estado de la tarea (plan/progreso/presupuesto, estructurado), memoria persistente
  (entre runs).
- **Salida estructurada:** esquema verificable mecánicamente. En este programa:
  `{kind, seed, result, evidence, limitations}` — evidencia y límites obligatorios.


### 🔩 El esqueleto en el laboratorio

| Pieza | En `run_lab("agent")` |
|---|---|
| Instrucciones | objetivo "verificar estado y sumar 7 + 5" + condición de parada |
| Herramientas | `status()` (lectura), `sum(left, right)` (pura) |
| Estado | `trace` + condiciones verificadas |
| Salida | contrato JSON con `evidence` y `limitations` |

Todo framework de agentes (Claude SDK, OpenAI Agents, LangGraph) es una implementación
opinada de este mismo esqueleto.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Bastan ~6 asserts: (1) claves de nivel superior `{kind, seed, result,
evidence, limitations}`; (2) `kind == "agent"`; (3) `evidence` lista no vacía; (4)
`limitations` lista no vacía; (5) cada paso de `trace` tiene `action` con `tool` (str) y
`args` (dict); (6) cada paso tiene `observation`. Que el contrato completo se valide con
media docena de asserts es exactamente el argumento a favor de la salida estructurada.

**Ejercicio 2.** Versión operativa posible — ROL/OBJETIVO: "procesas reportes de gastos;
éxito ⇔ cada gasto queda clasificado en una categoría del catálogo con su comprobante
adjunto validado". RESTRICCIONES: "nunca apruebas gastos; montos > 500 USD o sin
comprobante se marcan para revisión humana". PROCEDIMIENTO: "usa `ocr_comprobante` antes
de clasificar; si la categoría es ambigua, elige la más restrictiva y anótalo; detente
tras procesar todos los gastos o al paso 50". FORMATO: "JSON con `gastos[]` (id,
categoria, monto, estado), `pendientes_humano[]`, `evidence`, `limitations`".

**Ejercicio 3.** (a) contexto — se necesita para decidir ahora y muere con el run.
(b) estado de la tarea — progreso estructurado, permite reanudar. (c) memoria persistente
— sobrevive entre runs (o se inyecta a instrucciones). (d) estado de la tarea — contador
del run. (e) ambos: efecto en el entorno + registro en el estado de la tarea. (f) memoria
persistente — conocimiento acumulado entre runs (clase 115).

**Ejercicio 4.** Ver celda de código: lo esencial es que el veredicto sea un enum cerrado
(no prosa), que cada hallazgo tenga ubicación verificable (archivo, línea) — eso es
`evidence` en forma de datos —, y que `limitations` declare qué no se revisó (archivos
omitidos, tests no ejecutados) para que el consumidor no infiera cobertura total.


In [ ]:
result = run_lab("agent", seed=110)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — validador mecánico del contrato
result = run_lab("agent", seed=110)

def valida_contrato(r):
    assert set(r) == {"kind", "seed", "result", "evidence", "limitations"}
    assert r["kind"] == "agent"
    assert isinstance(r["evidence"], list) and r["evidence"]
    assert isinstance(r["limitations"], list) and r["limitations"]
    for paso in r["result"]["trace"]:
        assert isinstance(paso["action"]["tool"], str)
        assert isinstance(paso["action"]["args"], dict)
        assert "observation" in paso
    return True

print("contrato válido:", valida_contrato(result))


In [ ]:
# Ejercicio 4 — esquema de salida del agente revisor de PRs
esquema_salida = {
    "veredicto":       ("enum: aprobar | cambios_requeridos | no_concluyente",
                        "obligatorio", "decisión componible por CI"),
    "hallazgos":       ("lista de {archivo, linea, severidad, descripcion}",
                        "obligatorio", "evidence en forma de datos verificables"),
    "archivos_revisados": ("lista de str", "obligatorio", "cobertura real del run"),
    "evidence":        ("lista de str", "obligatorio", "hechos que sostienen el veredicto"),
    "limitations":     ("lista de str", "obligatorio",
                        "qué NO se revisó: tests no ejecutados, archivos omitidos"),
    "presupuesto":     ("{pasos_usados, pasos_max}", "obligatorio", "auditoría de costo"),
}
for campo, (tipo, obligatorio, uso) in esquema_salida.items():
    print(f"{campo:20s} {obligatorio:12s} {uso}")


## Reflexión

1. La separación "el modelo emite intenciones, el runtime ejecuta" parece un detalle de
   implementación. Nombra dos controles concretos (de clases 116-117) que serían
   imposibles si el modelo ejecutara herramientas directamente.
2. El contrato del laboratorio obliga a `evidence` y `limitations` en la salida. ¿Qué
   verificación mecánica permite cada campo, y qué se pierde si el resultado final del
   agente fuera prosa libre?
3. ¿Por qué "meter todo el historial al contexto" y "no registrar estado estructurado de
   la tarea" son el mismo error visto desde dos capas distintas del estado? ¿Qué operación
   (reanudar, auditar, cobrar) rompe cada uno?
